<!-- nav -->
[← 9 · When requirements change: evolving schemas](09_When_Requirements_Change.ipynb) · [Home](../../README.md) · [Schemas Framework design →](../../docs/FRAMEWORK.md)

# 10 · Asking the store: mutations as queries

**The problem.** A small team keeps its people in a store: who is on the roster, what each one does, and who knows
whom, since when. Now the questions start. Who are the engineers? Whom has Ada known since 2020? Who doesn't know Ada
yet? And then the changes that follow from the answers: give every lead a door badge, mark the people Ada mentors.

You could read the objects one by one and write loops. This case study asks the store instead, with the same thing
every write in the earlier case studies was made of: a *transaction*. A transaction is an expression, a chain of
*terms*, and it gives back what it found, as *frames*. Writing and asking are the same act, and that is the idea this
case study builds on:

1. A transaction as an expression, and the frames it gives back.
2. `map`: symbols that range over a schema's objects, a spec that filters them.
3. Following an adjacency, a keyed adjacency, a singleton, and a symbol given one object.
4. Absence as a filter.
5. `reduce`, which drops symbols, and `partition`, which forks the stream.
6. Writing from frames: an `update` or an `upsert` per frame.
7. Handing frames on, as references.
8. What it does not do yet.

The calls are the Python tutorial's, in TypeScript's form: a frame is a `Map` from each symbol to its object, an
integer is a `bigint` (`since(2020n)`), and the store and the builders are typed loosely, so every callback is annotated
`(x: any)` (case study 1).

In [1]:
import { Builders, Paths, References } from "@mbse/schemas/Framework";

/** The team: people who know each other, a roster owning them by key, and badges each person owns by key. */
function install(txn: any): any {
  const S = txn.Schemas; // the store's own schemas, by name
  return txn.uid("tutorial.team/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Holds: S.OfRelation, Roster: S.OfObject,
      Wears: S.OfRelation, Badge: S.OfObject })
    .spec([
      (_: any) => _.Knows.name("team.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]).properties({
        since: S.OfInt }),
      (_: any) => _.Person.name("team.Person").ref(true).properties({ name: S.OfStr, role: S.OfStr }).adjacencies([
        (t: any) => t.name("knows").relation([_.Knows]).me("knower"),
        (t: any) => t.relation([_.Knows]).me("known"),
        (t: any) => t.name("roster").relation([_.Holds]).me("member").owner(true),
      ]),
      (_: any) => _.Holds.name("team.Holds").links([(t: any) => t.name("roster"), (t: any) => t.name("member")]).properties({
        key: S.OfStr }).uniques([["roster", "key"], ["member"]]),
      (_: any) => _.Roster.name("team.Roster").ref(true).singleton("team.TheRoster").adjacencies({
        members: (t: any) => t.relation([_.Holds]).me("roster") }),
      (_: any) => _.Wears.name("team.Wears").links([(t: any) => t.name("person"), (t: any) => t.name("badge")]).properties({
        key: S.OfStr }).uniques([["person", "key"], ["badge"]]),
      (_: any) => _.Person.adjacencies({ badges: (t: any) => t.relation([_.Wears]).me("person") }),
      (_: any) => _.Badge.name("team.Badge").ref(true).properties({ access: S.OfStr }).adjacencies({
        wearer: (t: any) => t.relation([_.Wears]).me("badge").owner(true) }),
    ])]))]);
}

const store: any = new Builders.Store();
store.transaction(install);
const [Person, Roster, Badge] = ["team.Person", "team.Roster", "team.Badge"].map((name) => store.registered(name));
console.log([...store.module("team").schemas.keys()]);

[ "Knows", "Person", "Holds", "Roster", "Wears", "Badge" ]


The install defines six schemas in the module `team`. The roster is a singleton (case study 9) that *owns* each
person: a person's `roster` adjacency is declared `.owner(true)`, so the entry that links a person to the roster owns
them, and `team.Holds`' `uniques` make its `key` name one person in the roster. A person owns each of their badges
the same way, by a key of their own. An owned object has a *path*, its owner's and the key (case study 6), which
steps 3, 6 and 7 rely on.

## 1. A transaction as an expression

You have met the install's shape already: one *upsert*, whose pattern has *symbols* and a *spec*. The team's people are
written the same way. Each symbol stands for an object of its schema, and the spec says what must hold of them, with
the calls their builders take, made on the symbols: the roster's members, by key, each person's name and role, and who
knows whom. A symbol in a spec, `_.bob`, is the object it stands for.

`store.transaction(...)` runs the expression and gives back its *frames*: one per way the pattern was satisfied, each
mapping every symbol to the object it is bound to. An upsert from nothing has one frame, the objects it found or made:

In [2]:
/** Ada, Bob, Cy and Dee, on the roster, and who knows whom since when. */
function team(txn: any): any {
  return txn.uid("tutorial.team.people/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ roster: Roster, ada: Person, bob: Person, cy: Person, dee: Person })
    .spec([
      (_: any) => _.roster.members({ ada: (e: any) => e.member(_.ada), bob: (e: any) => e.member(_.bob),
        cy: (e: any) => e.member(_.cy), dee: (e: any) => e.member(_.dee) }),
      (_: any) => _.ada.name("Ada").role("lead").knows([
        (e: any) => e.known(_.bob).since(2020n), (e: any) => e.known(_.cy).since(2022n)]),
      (_: any) => _.bob.name("Bob").role("engineer").knows([(e: any) => e.known(_.cy).since(2021n)]),
      (_: any) => _.cy.name("Cy").role("engineer"),
      (_: any) => _.dee.name("Dee").role("designer").knows([_.ada]),
    ])]))]);
}

const frames = store.transaction(team);
console.log(frames);

[
  Map(5) {
    "roster" => <team.Roster>,
    "ada" => <team.Person>,
    "bob" => <team.Person>,
    "cy" => <team.Person>,
    "dee" => <team.Person>
  }
]


The frame binds `roster` to the store's one roster: a symbol whose schema is a singleton's binds the singleton. Each
person is found by the roster's entry that owns them, or made where there is none, so run again, the transaction finds
the same people and writes nothing.

A frame holds objects, and to read one you read its objects. `names` does that for the rest of this case study: each
frame, its symbols and the names of the people they bind.

In [3]:
const again = store.transaction(team); // run again: found, not made again
console.log([...frames[0].keys()].every((symbol: string) => again[0].get(symbol) === frames[0].get(symbol)));
const ada = frames[0].get("ada");

/** Each frame's symbols, with the names of the people they bind. */
function names(frames: Map<string, any>[]): Record<string, string>[] {
  return frames.map((frame) => Object.fromEntries([...frame].map(([symbol, person]) => [symbol, `${person.name}`])));
}

true


## 2. Map: symbols ranging over a schema's objects

An upsert finds or makes. A *map* only finds: each symbol it adds ranges over the objects of its schema the store holds,
one frame each. With no spec, that is everyone. With a spec, it is everyone of whom the spec holds, here a property's
value. Map and reduce are meant in MapReduce's sense, not as a list's `map`: a map works on each frame alone and may give
none, one or many, so that a transaction can be sharded and run in parallel; a reduce groups what remains and folds
each group ([Mutations as queries](../../docs/STREAMING_QUERIES.md)).

The pattern of a map is given in its `symbols`, and the term is chosen, as an upsert is, through `mu`:

In [4]:
const everyone = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: Person })]))]));
const engineers = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: Person }).spec([(_: any) => _.x.role("engineer")])]))]));
console.log(names(everyone));
console.log(names(engineers));

[ { x: "Ada" }, { x: "Bob" }, { x: "Cy" }, { x: "Dee" } ]


[ { x: "Bob" }, { x: "Cy" } ]


The map reaches what the store keeps: the objects its singletons reach, here the roster and everyone on it (case
study 9). A spec in a map filters: what an upsert would write where it does not hold, a map asks for.

## 3. Following an adjacency

With two symbols, a map's frames are every pair, and a spec that links them keeps the pairs it links: `_.x.knows([_.y])`
holds where `x` knows `y`. An entry can be constrained too, with the call a builder's entry takes:
`(e: any) => e.known(_.y).since(2020n)` holds where `x` has known `y` since 2020.

In [5]:
const pairs = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: Person, y: Person }).spec([(_: any) => _.x.knows([_.y])])]))]));
const since2020 = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: Person, y: Person }).spec([(_: any) => _.x.knows([(e: any) => e.known(_.y).since(2020n)])])]))]));
console.log(names(pairs));
console.log(names(since2020));

[
  { x: "Ada", y: "Bob" },
  { x: "Ada", y: "Cy" },
  { x: "Bob", y: "Cy" },
  { x: "Dee", y: "Ada" }
]


[ { x: "Ada", y: "Bob" } ]


A keyed adjacency takes a mapping, as it does when you write one: the roster's member under the key `"ada"`. The
roster's symbol binds the roster, being a singleton's, so this map starts from one object and follows one entry:

In [6]:
const keyed = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ roster: Roster, y: Person }).spec([(_: any) => _.roster.members({ ada: (e: any) => e.member(_.y) })])]))]));
console.log(keyed.map((frame: Map<string, any>) => `${frame.get("y").name}`));

[ "Ada" ]


To start from any one object, give its symbol a *reference* to it in place of a schema. `References.of(store, ada)`
designates Ada by her path, and a symbol given it binds her, and nothing else (or nothing, where the store holds
nothing at that path). Its spec filters, or is written, as any symbol's:

In [7]:
const knownByAda = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: References.of(store, ada), y: Person }).spec([(_: any) => _.x.knows([_.y])])]))]));
console.log(names(knownByAda));

[ { x: "Ada", y: "Bob" }, { x: "Ada", y: "Cy" } ]


## 4. Absence as a filter

A spec can say what must be absent as well. `remove` in a spec, `(a: any) => a.remove((e: any) => e.known(_.y))`, would
remove the entry where it is written; in a map, it holds where there is no such entry to remove. So this asks who
doesn't know Ada (Ada among them: she doesn't know herself):

In [8]:
const strangers = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: Person, y: References.of(store, ada) })
  .spec([(_: any) => _.x.knows((a: any) => a.remove((e: any) => e.known(_.y)))])]))]));
console.log(names(strangers));

[
  { x: "Ada", y: "Ada" },
  { x: "Bob", y: "Ada" },
  { x: "Cy", y: "Ada" }
]


## 5. Reduce and partition

Terms chain: each passes the frames it ends with to the term in its `then`, which may add symbols, drop them, write, or
fork. A *reduce* drops symbols, and keeps each frame left once. Dropping `y` from the pairs of step 3 leaves everyone
who knows someone, each once, though Ada knows two people. The reduce's *collector* names the pattern it reads and the
symbols it drops:

In [9]:
const knowers = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: Person, y: Person }).spec([(_: any) => _.x.knows([_.y])])])
  .then([(w: any) => w.reduce((r: any) => r.symbols([(c: any) => c
    .pattern([(p: any) => p.symbols({ x: Person, y: Person })]).symbols(["y"])]))]))]));
console.log(names(knowers));

[ { x: "Ada" }, { x: "Bob" }, { x: "Dee" } ]


A *partition* forks the stream into *branches*. Each frame goes through every branch, or through only the first or the
last (`first(true)`, `last(true)`), and a partition with no branch runs its `none`. What the branches end with is joined,
and passed to the partition's `then`, if it has one. Here one branch keeps who knows someone and the other who is known
by someone, in one transaction:

In [10]:
const both = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: Person, y: Person }).spec([(_: any) => _.x.knows([_.y])])])
  .then([(w: any) => w.partition((q: any) => q.branches([
    (e: any) => e.branch((b: any) => b.then([(w: any) => w.reduce((r: any) => r.symbols([(c: any) => c
      .pattern([(p: any) => p.symbols({ x: Person, y: Person })]).symbols(["y"])]))])),
    (e: any) => e.branch((b: any) => b.then([(w: any) => w.reduce((r: any) => r.symbols([(c: any) => c
      .pattern([(p: any) => p.symbols({ x: Person, y: Person })]).symbols(["x"])]))])),
  ]))]))]));
console.log(names(both));

[
  { x: "Ada" },
  { x: "Bob" },
  { x: "Dee" },
  { y: "Bob" },
  { y: "Cy" },
  { y: "Ada" }
]


A branch will be chosen by a condition it requires, a constraint over the frame. Constraints are expressions, which
this package does not evaluate (step 8), so for now a partition's branches have none: every branch runs, or the first,
or the last.

## 6. Writing from frames

What a map finds, the terms after it can write. An *update* writes its spec over each frame it is given, where the spec
does not hold already. Here everyone Ada has known since 2020 becomes her mentee:

In [11]:
store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ x: References.of(store, ada), y: Person })
  .spec([(_: any) => _.x.knows([(e: any) => e.known(_.y).since(2020n)])])])
  .then([(w: any) => w.update((u: any) => u.apply([(p: any) => p.symbols({ y: Person }).spec([(_: any) => _.y.role("mentee")])]))]))]));
console.log([...store.extent("team.Person")].map((person: any) => [`${person.name}`, `${person.role}`]));

[
  [ "Ada", "lead" ],
  [ "Bob", "mentee" ],
  [ "Cy", "engineer" ],
  [ "Dee", "designer" ]
]


An update's pattern names symbols a term before it bound; an update first in a transaction binds none, and the store
refuses it before writing anything.

An *upsert* after a map finds or makes an object per frame. A badge is owned by the person who wears it, under a key, so
an upsert finds it by that owning entry: run again, it finds the badge it made, and writes nothing. Every lead gets a
door badge:

In [12]:
/** A door badge for every lead, found by the lead's entry that owns it, or made. */
function badges(txn: any): any {
  return txn.uid("tutorial.team.badges/1").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ x: Person }).spec([(_: any) => _.x.role("lead")])])
    .then([(w: any) => w.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ x: Person, b: Badge })
      .spec([(_: any) => _.x.badges({ door: (e: any) => e.badge(_.b) }), (_: any) => _.b.access("every floor")])]))]))]);
}

const issued = store.transaction(badges);
console.log(issued.map((frame: Map<string, any>) => [`${frame.get("x").name}`, `${frame.get("b").access}`]));
const reissued = store.transaction(badges); // run again: the same badge, found by its owner and key
console.log(reissued[0].get("b") === issued[0].get("b"), [...ada.badges].length);

[ [ "Ada", "every floor" ] ]


true 1


## 7. Handing frames on

A frame is the store's answer, but a service usually answers another program, which holds none of the store's objects.
`References.of(store, frame)` gives a reference to a frame: a value of the store's, written and read as any value is,
that designates each symbol's object by its path. `References.resolve` gives back what it designates, here the frame
itself. Only objects with paths have references: a frame of owned objects and singletons has one, and a frame that
binds a *counted* object, which nothing names, has none.

In [13]:
const answer = References.of(store, issued[0]);
console.log(answer.as_schema().name);
const resolved = References.resolve(store, answer) as Map<string, any>;
console.log(resolved.size === issued[0].size && [...resolved].every(([symbol, target]) => issued[0].get(symbol) === target));

References.OfFrame


true


A frame reference can also carry what it witnesses: property values an object holds (`assigns`) and relation entries
(`entries`), each by its objects' references, so that the program it is handed to can see why the frame holds. This
one says that Ada is a lead and that she has known Bob since 2020. Resolved, it gives its bindings:

In [14]:
const bob = frames[0].get("bob");
const Knows = store.registered("team.Knows");
const witness = store.References.OfFrame((f: any) => f
  .binds({ ada: (o: any) => o.path(Paths.of(ada)), bob: (o: any) => o.path(Paths.of(bob)) })
  .assigns([(x: any) => x.as_object((a: any) => a.object((o: any) => o.path(Paths.of(ada))).properties({ role: "lead" }))])
  .entries([(x: any) => x.as_object((e: any) => e.relation(Paths.of(Knows)).links({
    knower: (o: any) => o.path(Paths.of(ada)), known: (o: any) => o.path(Paths.of(bob)) }).properties({ since: 2020n }))]));
console.log(names([References.resolve(store, witness) as Map<string, any>]));

[ { ada: "Ada", bob: "Bob" } ]


## 8. What it does not do yet

The edges are worth knowing:

- **Entries are reached through an object's named adjacency.** A spec says `_.x.knows(...)`: there is no term over a
  relation's whole extent, and no verb that writes a relation's entries but through an object.
- **No expressions.** A pattern's `requires` (constraints over the frame), its *variables* (values a frame assigns,
  rather than objects it binds), a map or a reduce of variables, and a reduce that *collects* (folds frames into one
  value) are all expressions, which mbse-expressions will evaluate. Until it is ported to these transactions, the store
  refuses each before it writes anything:

In [15]:
for (const term of [
  (mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.variables({ n: store.Schemas.OfInt })])),
  (mu: any) => mu.map((m: any) => m.variables([(p: any) => p.symbols({ x: Person })])),
]) {
  try {
    store.transaction((t: any) => t.mutations([term]));
  } catch (error) {
    console.log(String(error));
  }
}

ValueError: an upsert's pattern has variables: transactions bind no variables until expressions are evaluated


ValueError: a map of variables: transactions bind no variables until expressions are evaluated


A spec matches by equal values, then: `since(2020n)` and not "before 2021". That is what the expressions will add.

## What you learned

- A transaction is an expression: a chain of terms, each passing the frames it ends with to its `then`.
  `store.transaction(...)` runs it and gives back the frames its chain ends with, each a `Map` of symbols to objects.
- A `map` binds each symbol it adds to every object of its schema the store keeps, one frame each, and its spec
  filters them: by a property's value, by an adjacency's entries and their properties, by a keyed adjacency's mapping.
- A symbol of a singleton's schema binds the singleton, and a symbol given a reference, `References.of(store, ada)`,
  binds the one object it designates: the way to start from one object.
- A spec's `remove` holds where the entry is absent, so a map can ask what is missing.
- A `reduce` drops symbols, keeping each frame left once; a `partition` forks the stream into branches and joins what
  they end with.
- An `update` writes its spec over each frame; an `upsert` finds or makes an object per frame, an owned one by its
  owning entry, so run again it writes nothing.
- `References.of(store, frame)` hands a frame of owned objects on, as a value; a frame reference may carry its witness,
  the property values and entries that make it hold.
- Expressions (`requires`, variables, collecting reduces) are refused until mbse-expressions evaluates them.

**Next:** the [design document](../../docs/FRAMEWORK.md) is the reference for everything here, and
[mbse-expressions](https://github.com/pitaman71/mbse-expressions) continues where step 8 stops.

<!-- nav -->

<!-- nav -->
[← 9 · When requirements change: evolving schemas](09_When_Requirements_Change.ipynb) · [Home](../../README.md) · [Schemas Framework design →](../../docs/FRAMEWORK.md)